**** data setiyle ilgili kısa bir özet yaz

## Read.me

*  Veri 129.971 satır, 14 kolon. Kaynak: WineEnthusiast, Haziran 2017.
* ABD'nin 54.265 kaydı da toplam verinin %45'i. Dergi Amerikan olduğu için Amerikan şaraplarını çok daha yoğun değerlendirmiş.
*   Puanlar 80-100 arası. Dergi 80 altını yayınlamıyor → veri seçilmiş örneklem.
* Fiyat %6.9 eksik (8.996 kayıt). Ortalama 35$, ortanca 25$, maks 3.300$
* Fiyat-puan korelasyonu: ham 0.416, log 0.612.
* Fiyat her iki katına çıktığında puan ~1.7 artıyor (azalan getiri).
* En iyi fiyat/performans bandı: 15-25$.

#1. Keşifsel Veri Analizi (EDA)



## Datanın eklenmesi

import pandas as pd — pandas, Python'da tablo işlemek için kullanılan kütüphane. as pd kısmı ona kısa bir takma ad veriyor, böylece her seferinde pandas.birşey yazmak yerine pd.birşey yazıyoruz.

df = pd.read_csv(...) — CSV dosyasını okuyup df adlı bir değişkene koyuyor. df "dataframe" demek, yani tablo.

df.shape[0] ve [1] — tablonun kaç satır ve kaç kolon olduğunu veriyor.

df.head() — ilk 5 satırı güzel bir tablo halinde gösteriyor.

In [3]:
import pandas as pd

df = pd.read_csv("winemag-data-130k-v2.csv")

print("Satır sayısı:", df.shape[0])
print("Kolon sayısı:", df.shape[1])
df.head()

Satır sayısı: 46422
Kolon sayısı: 14


,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


## Datanın incelenmesi

df.info():kolon tipleri-- pandasta string'e object deniliyor


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46422 entries, 0 to 46421
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             46422 non-null  int64  
 1   country                46398 non-null  object 
 2   description            46422 non-null  object 
 3   designation            33112 non-null  object 
 4   points                 46422 non-null  int64  
 5   price                  43119 non-null  float64
 6   province               46398 non-null  object 
 7   region_1               38802 non-null  object 
 8   region_2               17825 non-null  object 
 9   taster_name            36982 non-null  object 
 10  taster_twitter_handle  35229 non-null  object 
 11  title                  46422 non-null  object 
 12  variety                46421 non-null  object 
 13  winery                 46421 non-null  object 
dtypes: float64(1), int64(2), object(11)
memory usage: 5.0+

her bir kolandaki eksik datanın sayısını ve sonra yüzdesini buluyoruz. yüzdeye göre o satırı silecek miyiz yoksa kalacak mı karar vereceğiz.

In [ ]:
eksik = df.isnull().sum().sort_values(ascending=False)
oran = (eksik / len(df) * 100).round(1)

pd.DataFrame({"eksik_adet": eksik, "eksik_yuzde": oran})

In [5]:
df[["points", "price"]].describe()

,points,price
count,46422.000000,43119.000000
mean,88.437314,35.333287
std,3.035138,40.114869
min,80.000000,4.000000
25%,86.000000,17.000000
50%,88.000000,25.000000
75%,91.000000,42.000000
max,100.000000,2500.000000


Çıkan sonuca göre karar verdiklerimiz:

1.  Sileceklerimiz:
*    region_2 %61 boş olduğu için siliyoruz. Ayrıca region_1 değerimiz olduğu için bizim için gerekli bir veri değil.
*    Unnamed: 0 satır numaraları, gereksiz veri
*    taster_twitter_handle biizm için gereksiz zaten taster_name ile tanıyoruz ama olmasa da sorun değil kişi bilgisi.
* country ve variety kolonlarındaki 63 ve 1 boş kayıt ise ihmal edilebilir. Bu satırları da sileceğiz.

2.   Değişiklik yapılmayacklar:

*    designation %29 şarabın özel bağ adı — her şarabın olmak zorunda değil, yokluğu doğal.
* region_1 %16 yine olmak zorunda değil
* taster_name %20 boş değrlendirici "bilinmiyor" diye işaretleyeceğiz.
3. Bizim için kritik olan eksiklikler
* price %6.9 boş — 8.996 şarap.Fiyat-puan ilişkisi de kuracağımız için bizim için önemli bir veri. Burayı ortalama vb ile doldurabiliriz fakat biz bu analizde bu satırları değiştirmeyeceğiz. Fiyat ilişkisi kurmak istediğimizde bu satırları dahil etmeyeceğiz.

Sayısal veriden elde ettiklerimiz:
* bu iyi şaraplarla ilgili bir derginin verisi olduğu için sadece iyi şaraplar listeleniyor. Yarı puanı 100 üzerinden 80 ile 100 arasındaki şarapları görebiliyoruz.
* Fiyat ortalaması 35, en ucuz şarap 4 en pahalı ise 3300. Ortalamanın düşük olması çok az sayıda şarabın pahalı olduğunu gösteriyor.

## Soru: Puan ve fiyat arasında ilişki var mı?



YÖNTEM 1 — Gruplama
Veriyi 5 fiyat aralığına böldük: 0-15, 15-25, 25-50, 50-100, 100-3300
Aralıklara BİZ karar verdik. Nasıl ki otel verisinde "500 euro üstü
pahalıdır" dediysek, burada da hem kendi alan bilgimize hem de
describe() çıktısındaki çeyrekliklere (17$, 25$, 42$) göre böldük.
Her aralıktaki şarap sayısını ve ortalama puanı hesaplattık.

Sonuç: 85.9 → 87.6 → 89.3 → 91.1 → 92.8
Fiyat arttıkça PUAN artıyor. Ama fiyat 200 kat artarken puan
sadece 7 artıyor. Yani ilişki var, güçlü değil.

YÖNTEM 2 — Korelasyon
Bu gruplamadan bağımsız, tüm satırlar üzerinden hesaplanıyor.
Ham fiyatla: 0.416 → orta düzey ilişki
İki yöntem de aynı şeyi söyledi: ilişki var ama zayıf.
(Farklı yollardan aynı cevap = bulgu sağlam.)

LOG DENEMESİ
Fiyat çok arttıkça puanın o kadar artmadığını gördük.
"İlişki logaritmik olabilir mi?" diye düşündük.
Önce fiyatların log değerini aldık, sonra yeni hesaplanmış her bir fiyatle puanın korelasyonunu
hesapladık: 0.612
Yükseldi → evet, ilişki logaritmik.
Yani her fiyat KATLANMASI ~1.7 puan getiriyor (azalan getiri).


In [ ]:
fiyatli = df[df["price"].notna()]

korelasyon = fiyatli ["price"].corr(fiyatli ["points"])
print("Fiyat-puan korelasyonu:", round(korelasyon, 3))

print()
print("Fiyat aralığına göre ortalama puan:")
fiyatli .groupby(pd.cut(fiyatli ["price"], [0, 15, 25, 50, 100, 3300]))["points"].agg(["mean", "count"]).round(2)

 şarap fiyatı olmayan veriyi işleme dahil etmeyeceğiz demiştik. fiyatli = df[df["price"].notna()] bu ifadeyle price sadece dolu olanlardan oluşan bir tablo oluşuyor

In [ ]:
import numpy as np

fiyatli = df[df["price"].notna()]

ham = fiyatli["price"].corr(fiyatli["points"])
log = np.log(fiyatli["price"]).corr(fiyatli["points"])

print("Ham fiyat ile korelasyon:", round(ham, 3))
print("Log fiyat ile korelasyon:", round(log, 3))

## Görselleştirmeler

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (10, 5)

fiyatli = df[df["price"].notna()]
print(len(fiyatli), "şarapta fiyat var")

In [ ]:
plt.hist(fiyatli["price"], bins=100)
plt.title("Fiyat dağılımı")
plt.xlabel("Fiyat ($)")
plt.ylabel("Şarap sayısı")
plt.show()

In [ ]:
#logaritmasını alınca daha anlamlı bir grafik çıkıyor
plt.hist(np.log(fiyatli["price"]), bins=100)
plt.title("Log fiyat dağılımı")
plt.xlabel("log(Fiyat)")
plt.ylabel("Şarap sayısı")
plt.show()

puan dağılımı: bins=21 çünkü puanlar 80-100 arası, yani 21 farklı değer var.
Burada simetrik bir tepe göreceksin, ortası 88 civarı. Hatırla, bu derginin 80 altını yayınlamamasından kaynaklanıyor

In [ ]:
plt.hist(df["points"], bins=21)
plt.title("Puan dağılımı")
plt.xlabel("Puan")
plt.ylabel("Şarap sayısı")
plt.show()

sample(5000) — 120 bin noktayı çizmek hem yavaş hem de okunmaz bir leke üretir. Rastgele 5 bin tanesini alıyoruz.
random_state=42 — rastgeleliği sabitliyor. Kodu tekrar çalıştırdığında aynı 5 bin şarap seçilir, grafiğin değişmez. Tekrarlanabilirlik için önemli, bir alışkanlık haline getir.
alpha=0.3 — noktaları yarı saydam yapıyor. Üst üste binen yerler koyulaşıyor, yoğunluğu görüyorsun.
xscale("log") — x eksenini log ölçeğe çeviriyor.

In [ ]:
ornek = fiyatli.sample(5000, random_state=42)

plt.scatter(ornek["price"], ornek["points"], alpha=0.3, s=10)
plt.xscale("log")
plt.title("Fiyat ve puan ilişkisi")
plt.xlabel("Fiyat ($, log ölçek)")
plt.ylabel("Puan")
plt.show()

# Kategorik keşif

1. Kaç farklı değer var

In [ ]:
kategorik = ["country", "province", "region_1", "variety", "winery", "taster_name"]

for kolon in kategorik:
    print(f"{kolon:15} {df[kolon].nunique():>6} farklı değer")

country             43 farklı değer
province           425 farklı değer
region_1          1229 farklı değer
variety            707 farklı değer
winery           16757 farklı değer
taster_name         19 farklı değer


value_counts() her değerin kaç kez geçtiğini sayıp çoktan aza sıralıyor.

In [ ]:
print("EN ÇOK ŞARAP ÜRETEN ÜLKELER")
print(df["country"].value_counts().head(10))

print("\nEN YAYGIN ÜZÜM ÇEŞİTLERİ")
print(df["variety"].value_counts().head(10))

print("\nEN ÇOK DEĞERLENDİRME YAPANLAR")
print(df["taster_name"].value_counts().head(10))

EN ÇOK ŞARAP ÜRETEN ÜLKELER
country
US           54504
France       22093
Italy        19540
Spain         6645
Portugal      5691
Chile         4472
Argentina     3800
Austria       3345
Australia     2329
Germany       2165
Name: count, dtype: int64

EN YAYGIN ÜZÜM ÇEŞİTLERİ
variety
Pinot Noir                  13272
Chardonnay                  11753
Cabernet Sauvignon           9472
Red Blend                    8946
Bordeaux-style Red Blend     6915
Riesling                     5189
Sauvignon Blanc              4967
Syrah                        4142
Rosé                         3564
Merlot                       3102
Name: count, dtype: int64

EN ÇOK DEĞERLENDİRME YAPANLAR
taster_name
Roger Voss            25514
Michael Schachner     15134
Kerin O’Keefe         10776
Virginie Boone         9537
Paul Gregutt           9532
Matt Kettmann          6332
Joe Czerwinski         5147
Sean P. Sullivan       4966
Anna Lee C. Iijima     4415
Jim Gordon             4177
Name: count, dtype: int64

ülkere göre puan ve fiyat
sayi >= 500 filtresi koyduk. Neden: 12 şarabı olan bir ülke tesadüfen 94 ortalama yapabilir ve listenin başına oturur. Az örneklemli gruplar yanıltıcıdır. Bu filtreyi koymak analizde önemli bir refleks.

In [ ]:
ulke_ozet = (
    fiyatli.groupby("country")
    .agg(sayi=("points", "size"), ort_puan=("points", "mean"), ort_fiyat=("price", "median"))
    .round(2)
)

ulke_ozet = ulke_ozet[ulke_ozet["sayi"] >= 500].sort_values("ort_puan", ascending=False)
ulke_ozet

,sayi,ort_puan,ort_fiyat
country,,,
Austria,2799,90.19,25.0
Germany,2120,89.84,26.0
France,17776,88.73,25.0
Italy,16914,88.62,28.0
Australia,2294,88.60,21.0
US,54265,88.57,30.0
Portugal,4875,88.32,16.0
New Zealand,1378,88.31,20.0
South Africa,1293,87.83,19.0


Üzüm çeşitlerine göre

In [ ]:
uzum_ozet = (
    fiyatli.groupby("variety")
    .agg(sayi=("points", "size"), ort_puan=("points", "mean"), ort_fiyat=("price", "median"))
    .round(2)
)

uzum_ozet[uzum_ozet["sayi"] >= 300].sort_values("ort_puan", ascending=False).head(15)

,sayi,ort_puan,ort_fiyat
variety,,,
Sangiovese Grosso,589,90.80,60.0
Nebbiolo,2331,90.33,56.0
Grüner Veltliner,1145,90.02,22.0
Port,499,90.01,40.0
Champagne Blend,1211,89.65,55.0
Riesling,4972,89.44,22.0
Pinot Noir,12787,89.41,42.0
Syrah,4086,89.29,35.0
Rhône-style Red Blend,1405,89.14,30.0


# 2.Temizlik---ödeve eklenmedi

csv'yi temzileyip bigquery atmak daha kolay ama gerçek hayatta genelde temizlenemmiş veri data warehouselarda olur. onun için burayı sadece pythonda nasıl yapılır öğrenmek için ekliyorum:

asıl datayı bozmamak için wine ismiyle yedeğini aldık:

In [ ]:
wine = df.copy()
print("Başlangıç:", wine.shape)

Başlangıç: (129971, 14)


daha önce belirlediğimiz gereksiz kolonları silelim:

In [ ]:
wine = wine.drop(columns=["Unnamed: 0", "region_2", "taster_twitter_handle"])
print(wine.shape)

(129971, 11)


boşlıkları kaldıralım

In [ ]:
metin_kolonlari = wine.select_dtypes(include="object").columns

for kolon in metin_kolonlari:
    wine[kolon] = wine[kolon].str.strip()

print("Kırpıldı:", list(metin_kolonlari))

Kırpıldı: ['country', 'description', 'designation', 'province', 'region_1', 'taster_name', 'title', 'variety', 'winery']


daha önce belirlediğmiz sileceğimiz 64 satır vardı onları siliyoruz:

In [ ]:
onceki = len(wine)
wine = wine.dropna(subset=["country", "variety"])
print(f"{onceki} → {len(wine)}  ({onceki - len(wine)} satır atıldı)")

129971 → 129907  (64 satır atıldı)


boş metinleri bilinmiyor vb dolduralım:

In [ ]:
wine["designation"] = wine["designation"].fillna("Bilinmiyor")
wine["region_1"] = wine["region_1"].fillna("Bilinmiyor")
wine["taster_name"] = wine["taster_name"].fillna("Anonim")

print(wine[["designation", "region_1", "taster_name"]].isnull().sum())

designation    0
region_1       0
taster_name    0
dtype: int64


son olarak yıl bilgisini çekelim ve ayrı bir kolon yapalım

In [ ]:
wine["vintage"] = wine["title"].str.extract(r"(\b(?:19|20)\d{2}\b)")
wine["vintage"] = pd.to_numeric(wine["vintage"])

print("Yıl bulunamayan:", wine["vintage"].isnull().sum())
print(wine["vintage"].describe())

wine.head(3)
wine.info()

Yıl bulunamayan: 4624
count    125283.000000
mean       2010.669660
std           3.797799
min        1904.000000
25%        2009.000000
50%        2011.000000
75%        2013.000000
max        2017.000000
Name: vintage, dtype: float64
<class 'pandas.core.frame.DataFrame'>
Index: 129907 entries, 0 to 129970
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   country      129907 non-null  object 
 1   description  129907 non-null  object 
 2   designation  129907 non-null  object 
 3   points       129907 non-null  int64  
 4   price        120915 non-null  float64
 5   province     129907 non-null  object 
 6   region_1     129907 non-null  object 
 7   taster_name  129907 non-null  object 
 8   title        129907 non-null  object 
 9   variety      129907 non-null  object 
 10  winery       129907 non-null  object 
 11  vintage      125283 non-null  float64
dtypes: float64(2), int64(1), object(9)
memory usage: 12

In [ ]:
wine[wine["vintage"] < 1990][["title", "vintage", "points"]].head(20)

,title,vintage,points
7986,Guidi 1929 2012 Aurea Riserva (Vernaccia di S...,1929.0,85
9728,1912 Winemakers 2011 Vinhas Velhas Branco Whit...,1912.0,90
12530,C. von Nell-Breuning 1976 Kaseler Dominikanerb...,1976.0,88
12860,Wiese & Krohn 1964 Branco Colheita (Port),1964.0,91
16011,Kirchmayr 1983 Solist White (Thermenregion),1983.0,89
16331,Taylor Fladgate 1967 Single Harvest (Port),1967.0,94
18792,Salon 1988 Le Mesnil Blanc de Blancs Brut Char...,1988.0,93
31417,Blandy's 1980 White (Madeira),1980.0,95
31425,Cossart Gordon 1987 Bual (Madeira),1987.0,95
33974,Argyle 1989 Extended Tirage Brut (Willamette ...,1989.0,86


# 3. BigQuery'ye Yükleme

aslında eklemeyi python yerine bigquery üzerinden kolayca yapabilirdik ama akışın tekrarlanabilir ve versiyonlanabilir olması için buradan yaptım. Sandbox'ta tablolar 60 günde siliniyor; script sayesinde tek komutla yeniden üretilebiliyor. yükleme için ayrıca Fivetran kullanılabilirdi ama bizim tek bir csv'miz var bunun için gereksiz uzun bir proses olacaktı.

In [ ]:
#Kimlik doğrulama
from google.colab import auth
auth.authenticate_user()
print("Giriş tamam")


Giriş tamam


In [ ]:
# Ayarlar
PROJECT_ID = "wine-analytics-508220"
DATASET = "wine_raw"
LOCATION = "europe-west1"
TABLE = "wine_reviews_raw"

In [ ]:
# asıl veride birkaç değişiklik yaptık. unnamed: 0 sütununu sildik ve kolon isimlerini küçük harfe çevirdik.
ham_veri = pd.read_csv("winemag-data-130k-v2.csv")
ham_veri = ham_veri.drop(columns=["Unnamed: 0"])
ham_veri.columns = [k.strip().lower() for k in ham_veri.columns]

print(ham_veri.shape)
print(list(ham_veri.columns))

(129971, 13)
['country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']


In [ ]:
# yükleme
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)
hedef = f"{PROJECT_ID}.{DATASET}.{TABLE}"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True,
)

job = client.load_table_from_dataframe(ham_veri, hedef, job_config=job_config, location=LOCATION)
job.result()

print("Yüklendi")

Yüklendi


In [ ]:
#kontrol
tablo = client.get_table(hedef)
print("Satır:", tablo.num_rows)
print("Kolon:", len(tablo.schema))
print()
for alan in tablo.schema:
    print(f"  {alan.name:15} {alan.field_type}")

Satır: 129971
Kolon: 13

  country         STRING
  description     STRING
  designation     STRING
  points          INTEGER
  price           FLOAT
  province        STRING
  region_1        STRING
  region_2        STRING
  taster_name     STRING
  taster_twitter_handle STRING
  title           STRING
  variety         STRING
  winery          STRING
